In [ ]:
!pip install -q -U langchain langchain-community langchain-text-splitters faiss-cpu sentence-transformers transformers


from langchain_community.document_loaders import CSVLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain.llms import HuggingFacePipeline


import nltk                    
nltk.download('punkt', quiet=True)   


import os, math, random, textwrap  
import numpy as np                  
import pandas as pd                 
from typing import List, Dict      


try:
    
    from langchain_community.document_loaders import CSVLoader, TextLoader, WikipediaLoader
    from langchain_community.vectorstores import FAISS
    from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
except Exception:

    from langchain.document_loaders import CSVLoader, TextLoader
    try:
        from langchain.document_loaders import WikipediaLoader
    except Exception:
        WikipediaLoader = None
    from langchain.vectorstores import FAISS
    try:
        from langchain.embeddings import HuggingFaceEmbeddings
    except Exception:
        pass
    try:
        from langchain.llms import HuggingFacePipeline
    except Exception:
        pass

from langchain.text_splitter import RecursiveCharacterTextSplitter  
from langchain.chains import RetrievalQA                              


from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM, pipeline

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction


SEED = 42
random.seed(SEED); np.random.seed(SEED)

print("Setup complete.")


In [ ]:
!pip install wikipedia
DATA_PATH = "wiki_sample.csv"  

docs = []   

if os.path.exists(DATA_PATH):
    
    loader = CSVLoader(DATA_PATH)         
    docs = loader.load()                   
    source_info = f"Loaded {len(docs)} rows from CSV."
elif 'WikipediaLoader' in globals() and WikipediaLoader is not None:

    loader = WikipediaLoader(query="Artificial intelligence", load_max_docs=3)
    docs = loader.load()
    source_info = f"Loaded {len(docs)} wiki pages via WikipediaLoader."
else:
   
    from langchain.schema import Document
    sample_texts = [
        "Machine learning is a subset of artificial intelligence focused on algorithms that learn patterns from data.",
        "Deep learning uses neural networks with many layers to model complex representations and achieve state-of-the-art results."
    ]
    docs = [Document(page_content=t, metadata={"source":"inline"}) for t in sample_texts]
    source_info = "Using tiny inline sample (offline fallback)."

print("Source:", source_info)
print("First doc snippet:", docs[0].page_content[:120], "...")


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,          
    chunk_overlap=100,       
    separators=["\n\n", "\n", " ", ""],  
)

splits = text_splitter.split_documents(docs)  # List[Document] of chunks
print(f"Created {len(splits)} chunks.")


for i, d in enumerate(splits[:5]):
    print(f"--- Chunk {i+1} (len={len(d.page_content)}) ---")
    print(textwrap.shorten(d.page_content, width=200, placeholder=" ..."))


In [ ]:


EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2" 


embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)


vector_store = FAISS.from_documents(splits, embeddings)


query = "What is machine learning?"
results = vector_store.similarity_search(query, k=2) 
print("\nQuery:", query)
for i, r in enumerate(results, start=1):
    print(f"#{i} (len={len(r.page_content)}):", textwrap.shorten(r.page_content, width=180, placeholder=" ..."))


In [ ]:

GEN_MODEL = "google/flan-t5-small"

task = "text2text-generation" if ("t5" in GEN_MODEL or "flan" in GEN_MODEL) else "text-generation"

hf_pipe = pipeline(task=task, model=GEN_MODEL)

llm = HuggingFacePipeline(pipeline=hf_pipe)

# Smoke test
prompt = "Explain artificial intelligence in one sentence."
print("Prompt:", prompt)
print("Output:", llm(prompt))


In [ ]:

retriever = vector_store.as_retriever(search_kwargs={"k": 3})  

qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff"    

sample_queries = [
    "What is deep learning?",
    "Define machine learning.",
    "How do neural networks learn?",
    "What is the role of embeddings?",
    "Explain the purpose of a vector database."
]

print("🔗 RAG Responses:")
for q in sample_queries:
    ans = qa.invoke(q)         
    print("\nQ:", q)
    print("A:", ans["result"])


In [ ]:

def rebuild_pipeline(chunk_size=700, chunk_overlap=120, k=4, embed_model=EMBED_MODEL):
    """Rebuilds the RAG pipeline with different chunking, k, or embedding model."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""]
    )
    chunks = splitter.split_documents(docs)
    _emb = HuggingFaceEmbeddings(model_name=embed_model)
    _vs = FAISS.from_documents(chunks, _emb)
    _retr = _vs.as_retriever(search_kwargs={"k": k})
    _qa = RetrievalQA.from_chain_type(llm=llm, retriever=_retr, chain_type="stuff")
    return _qa, _vs, chunks

qa2, vs2, chunks2 = rebuild_pipeline(chunk_size=700, chunk_overlap=120, k=4)

test_queries = ["What is deep learning?", "What is machine learning?"]

print("After refinement:")
for q in test_queries:
    print("\nQ:", q)
    print("A:", qa2.invoke(q)["result"])

print(f"\nOld #chunks: {len(splits)} | New #chunks: {len(chunks2)}")


In [ ]:

smooth = SmoothingFunction().method1  

eval_queries = [
    "What is deep learning?",
    "Define machine learning.",
    "How do neural networks learn?",
    "What is the role of embeddings?",
    "Explain the purpose of a vector database."
]

references = {
    "What is deep learning?": "Deep learning is a subset of machine learning that uses multi-layer neural networks to learn hierarchical representations.",
    "Define machine learning.": "Machine learning is a field of AI where algorithms learn patterns from data to make predictions or decisions without explicit rules.",
    "How do neural networks learn?": "Neural networks learn by adjusting weights via backpropagation and gradient descent to minimize a loss function on training data.",
    "What is the role of embeddings?": "Embeddings map items like words or passages to dense vectors so that semantic similarity corresponds to geometric proximity.",
    "Explain the purpose of a vector database.": "A vector database stores embeddings and supports fast similarity search to retrieve semantically relevant items."
}

rows = []
for q in eval_queries:
    generated = qa.invoke(q)["result"]
    ref = references[q]
    bleu = sentence_bleu([ref.split()], generated.split(), smoothing_function=smooth) 
    rows.append({"query": q, "generated": generated, "reference": ref, "bleu": bleu})

bleu_df = pd.DataFrame(rows)
bleu_df


In [ ]:

OPT_EMBED_MODEL = "sentence-transformers/all-mpnet-base-v2"  

qa_opt, vs_opt, chunks_opt = rebuild_pipeline(
    chunk_size=700, chunk_overlap=120, k=5, embed_model=OPT_EMBED_MODEL
)

opt_queries = [
    "What is deep learning?",
    "Define machine learning.",
    "What is the role of embeddings?"
]

rows2 = []
for q in opt_queries:
    gen = qa_opt.invoke(q)["result"]
    ref = references[q]
    bleu = sentence_bleu([ref.split()], gen.split(), smoothing_function=smooth)
    rows2.append({"query": q, "generated_opt": gen, "reference": ref, "bleu_opt": bleu})

bleu_opt_df = pd.DataFrame(rows2)
bleu_opt_df
